In [1]:
%cd ..

C:\Users\namtv40\Projects\prefecthq-external-ingestion\ingestions


In [2]:
from dotenv import load_dotenv

load_dotenv(".env.minio")


True

In [3]:
import sys
import os

sys.path.insert(0, r"C:\Users\namtv40\Projects\prefecthq-external-ingestion\ingestions")

In [4]:
import os
import sys
import json
import time
import requests
import pandas as pd
import pyarrow as pa
from dateutil import parser
import pyarrow.parquet as pq
from datetime import datetime


In [5]:

if not os.path.exists("./tmp/data"):
    os.makedirs("./tmp/data")


In [6]:
HDFS_BASE = "s3a://vcs-raw/crm-raw"
HIVE_DB = "crm_raw"

In [7]:
from common.config import *
from common.http_util import *
from common.crawler_util import *
from common.ambari_util import *


def fetch_resource_name_freshwork(resource_name, records, **kwargs):
    print("Start crawl : ", resource_name)
    start_time = time.time()

    
    # STATE_PATH = rc["state_path"]
    # BASE_URL = None
    # RESOURCE_URL = None
    # API_KEY_PATH = rc["api_key_path"]
    # API_COOKIE_PATH = rc["api_cookie_path"]
    # QUERY_PARAMS = None
    ENABLE_STATE =False
    # crawl_mode = "modified_and_new"
    crawl_mode = kwargs.get("crawl_mode", "static")
    schema_local_path = None
    # result_json_key = "deleted_deals"

    print(resource_name)
    start_time = time.time()
    # =========================
    # MAIN
    # =========================
    if ENABLE_STATE:
        last_state = read_last_state(resource_name)
        print("Last state =", last_state)
    else:
        last_state = None

    if not records:
        print("No new data")
        out_of_data = True
        return True

    # =========================
    # Pandas → Parquet
    # =========================

    now = datetime.now()
    partition_path = "{}/{}".format(HDFS_BASE, resource_name)

    filename = "data_{}_{}{:02d}{:02d}_{}{:02d}{:02d}.parquet".format(
        resource_name, now.year, now.month, now.day, now.hour, now.minute, now.second
    )
    local_parquet = "./tmp/data/{}/{}/{}".format(HIVE_DB, resource_name, filename)
    os.makedirs(os.path.dirname(local_parquet), exist_ok=True)

    # df.to_parquet(local_parquet,engine="pyarrow", compression="snappy", index=False)
    records = convert_json_add_ts_columns(records)

    schema_tm_path = "./resources/parquet_schema/{}/{}.json".format(HIVE_DB, resource_name)
    print(f"schema_tm_path={schema_tm_path}")
    schema = None
    if schema_local_path:
        schema = load_pyarrow_schema_from_json(schema_local_path)

    if os.path.exists(schema_tm_path):
        schema = load_pyarrow_schema_from_json(schema_tm_path)

    if not schema:
        schema = infer_schema_from_json(records)
        schema_json = save_pyarrow_type_to_json(schema)
        # os.makedirs(os.path.dirname(schema_tm_path), exist_ok=True)
        write_file_json(schema_tm_path, schema_json)

    records = convert_json_list_by_arrow_schema(records, schema)

    df = pd.DataFrame(records)
    data_table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)

    pq.write_table(data_table, local_parquet, compression="snappy")

    if crawl_mode == "static":
        replace_hdfs_https("{}".format(partition_path), local_parquet)
    else:
        upload_hdfs_https("{}".format(partition_path), local_parquet)

    print("Uploaded parquet to", partition_path)

    # =========================
    # Generate SQL (TEXT ONLY)
    # =========================
    sql = gen_spark_create_table(
        schema=schema,
        db=HIVE_DB,
        table=resource_name,
        location="{}/{}".format(HDFS_BASE, resource_name),
    )

    # filename = "create_table_{}_{}{:02d}{:02d}.sql".format(
    #     resource_name,
    #     now.hour,
    #     now.minute,
    #     now.second
    # )

    filename = "create_table_{}.sql".format(resource_name)

    local_sql = "./tmp/data/{}/{}/{}".format(HIVE_DB, resource_name, filename)
    os.makedirs(os.path.dirname(local_sql), exist_ok=True)

    with open(local_sql, "w") as f:
        f.write(sql)

    # upload_hdfs_https(
    #     "{}/{}".format(HDFS_BASE, resource_name),
    #     local_sql
    # )

    print("Uploaded SQL definition")

    return False


In [8]:
import pandas as pd
import re
from pathlib import Path
from collections import defaultdict

import re
from collections import defaultdict
def excel_col_name(idx: int) -> str:
    """Zero-based index → Excel column (A, B, ..., AA)"""
    name = ""
    while idx >= 0:
        idx, rem = divmod(idx, 26)
        name = chr(rem + ord("A")) + name
        idx -= 1
    return name


def snake_case(text: str) -> str:
    text = (
        str(text)
        .strip()
        .lower()
        .replace("\n", " ")
    )
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text

def read_csv_and_normalize_columns(
    file_path: str,
    mapping: dict,
    header_row=0,
    drop_rows=0
) -> pd.DataFrame:
    # đọc raw, chưa set header
    df = pd.read_csv(file_path, header=None, encoding="utf-8", dtype=str)
    raw_headers = df.iloc[header_row]
    seen = defaultdict(int)
    new_columns = []

    for idx, col in enumerate(raw_headers):
        if pd.isna(col) or col == "":
            base = "nan"
        else:
            # 1️⃣ ưu tiên dictionary
            base = mapping.get(str(col).strip())

            # 2️⃣ fallback snake_case
            if base is None:
                base = snake_case(col)
                print(col)

        seen[base] += 1

        if seen[base] > 1:
            excel_col = excel_col_name(idx).lower()
            base = f"{base}__{excel_col}"

        new_columns.append(base)

    # gán columns sạch
    df.columns = new_columns

    # drop header rows
    df = df.iloc[drop_rows:].reset_index(drop=True)
    df = df.fillna("")

    return df

def read_folder_and_union_csv(
    folder_path: str,
    mapping: dict,
    header_row=0,
    drop_rows=0,
    encoding="utf-8"
):
    dfs = []
    folder = Path(folder_path)

    files = sorted(folder.glob("*.csv"))
    if not files:
        raise ValueError("❌ Không tìm thấy file CSV nào trong folder")

    for file in files:
        df = read_csv_and_normalize_columns(
            file_path=file,
            mapping=mapping,
            drop_rows=drop_rows,
            header_row=header_row
        )

        # trace file nguồn
        df["source_file"] = file.name
        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)

In [9]:
import pandas as pd
import re
from collections import defaultdict
def excel_col_name(idx: int) -> str:
    """Zero-based index → Excel column (A, B, ..., AA)"""
    name = ""
    while idx >= 0:
        idx, rem = divmod(idx, 26)
        name = chr(rem + ord("A")) + name
        idx -= 1
    return name


def snake_case(text: str) -> str:
    text = (
        str(text)
        .strip()
        .lower()
        .replace("\n", " ")
    )
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text

def read_excel_and_normalize_columns(
    file_path: str,
    mapping: dict,
    sheet_name=0,
    header_row=0,
    drop_rows=0
) -> pd.DataFrame:
    # đọc raw, chưa set header
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=None, dtype=str)

    raw_headers = df.iloc[header_row]
    seen = defaultdict(int)
    new_columns = []

    for idx, col in enumerate(raw_headers):
        if pd.isna(col) or col == "":
            base = "nan"
        else:
            # 1️⃣ ưu tiên dictionary
            base = mapping.get(col)

            # 2️⃣ fallback snake_case
            if base is None:
                print(f"Not found for col = '{col}'")
                base = snake_case(col)

        seen[base] += 1

        if seen[base] > 1:
            excel_col = excel_col_name(idx).lower()
            base = f"{base}__{excel_col}"

        new_columns.append(base)

    # gán columns sạch
    df.columns = new_columns

    # drop header rows
    df = df.iloc[drop_rows:].reset_index(drop=True)

    return df



In [10]:
import re
import pandas as pd

def normalize_cell_text(value):
    if pd.isna(value):
        return value

    value = str(value).strip()                 # bỏ khoảng trắng đầu/cuối
    value = re.sub(r"\s+", " ", value)         # nhiều space -> 1 space
    value = value.title()                      # Bộ quốc phòng -> Bộ Quốc Phòng

    return value

def normalize_trim_text(value):
    if pd.isna(value):
        return value

    value = str(value).strip()                 # bỏ khoảng trắng đầu/cuối
    value = re.sub(r"\s+", " ", value)         # nhiều space -> 1 space
    return value


In [11]:

REVENUE_MAPPING = {
    "Ngày": "invoice_date",

    # ===== Time =====
    "Ngày xuất hóa đơn": "invoice_date",
    "Tháng": "report_month",

    # ===== Invoice info =====
    "Xuất hóa đơn (HD)/ tạm tính (TT)": "billing_type",
    "Nội dung đề nghị TT(Theo chứng từ gốc)": "transaction_description",
    "Nội dung đề nghị TT\n(Theo chứng từ gốc)": "transaction_description",

    # ===== Revenue =====
    "Doanh thu": "revenue_amount",
    "Chia sẻ": "shared_revenue_amount",
    "chia sẻ": "shared_revenue_amount",
    "Doanh thu cuối": "actual_revenue_amount",
    "Triệu đồng": "actual_revenue_amount",

    "VAT": "vat_amount",
    "Tiền hàng (đã bao gồm VAT)": "gross_amount",
    "Tiền hàng \n(đã bao gồm VAT)": "gross_amount",
    "Tiền hàng": "gross_amount",

    # ===== Revenue recognition =====
    "DT dòng tiền đều/ DT lên 1 lần": "revenue_type",

    # ===== Product / Service =====
    "Phân loại SP/DV": "service_category",
    "SPDV cụ thể": "service_offering_name",
    "SPDV": "service_offering_name",
    "Mã SPDV": "service_offering_code",

    # ===== Customer / Market =====
    "Khoản mục SPDV": "customer_segment_l1",                  # chưa có trong COLUMN_DICT
    "Nội bộ/ Ngoài/QTế/Thị Trường": "customer_segment_l1",    # chưa có trong COLUMN_DICT
    "Phân loại KH": "customer_classification",
    "Kênh khách hàng": "customer_channel",
    "Kênh khách hàng ": "customer_channel",
    "Khách hàng": "customer_name",
    "Segment": "segment",
    "Nhóm khách hàng": "customer_group",

    # ===== Org =====
    "Phòng": "department_name",
    "AM": "account_manager_name",
    "AM hiện tại": "account_manager_name",
    "AM hiện tại ": "account_manager_name",
    "Presale": "presale_name",

    # ===== Classification =====
    "Phân loại DT Cũ/Mới": "revenue_classification",
    "Phân loại SOC (SOC và non-SOC)": "soc_classification",

    # ===== Revenue sharing =====
    "DT chia sẻ từ MSS": "shared_revenue_from_mss_amount",

    # ===== Region =====
    "Bắc/Nam/Thị trường": "region_group",   # chưa có trong COLUMN_DICT
    "Thị trường ": "market_segment",
    "Nam/ Bắc": "business_center",

    # ===== Flags =====
    "Vvip": "is_vvip_customer",             # chưa có trong COLUMN_DICT
    "HĐ khung": "is_master_contract",       # chưa có trong COLUMN_DICT
    "HĐ khung ": "is_master_contract",
    "VIP/ VVIP theo TTr":"vvip_segment",

    # ===== Notes =====
    "Ghi chú": "note",                      # chưa có trong COLUMN_DICT
}

filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\fin-data-vudt\spdv.xlsx"
resource_name = "revenue_reconciliation"

df = read_excel_and_normalize_columns(
      filename,
    sheet_name="Doanh thu spdv",
    mapping=REVENUE_MAPPING,
    header_row=0,
    drop_rows=1,
)
df["source_file"] = r'finance-data\fin-data-vudt\spdv.xlsx'
df['vat_amount'] = df['vat_amount'].fillna('0')
df['shared_revenue_amount'] = df['shared_revenue_amount'].fillna('0')
df['gross_amount'] = df['gross_amount'].fillna('0')
df['revenue_amount'] = df['revenue_amount'].fillna('0')
df['actual_revenue_amount'] = df['actual_revenue_amount'].fillna('0')
df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce")
# Loại bỏ các bản ghi năm 2026
df = df[~df["invoice_date"].dt.year.isin([2024, 2025, 2026])]

df["report_month"] = df["report_month"].fillna( df["invoice_date"].dt.strftime("%m"))

exclude_cols = ["gross_amount", "source_file", "vat_amount","shared_revenue_amount" , "revenue_amount", "report_month", "actual_revenue_amount"]
cols = df.columns.difference(exclude_cols)
df[cols] = df[cols].fillna("")

df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")

df["customer_segment_l1"] = df["customer_segment_l1"].apply(normalize_cell_text)
# df["business_center"] = df["business_center"].apply(normalize_cell_text)
df["market_segment"] = df["market_segment"].apply(normalize_cell_text)
df["segment"] = df["segment"].apply(normalize_cell_text)
df["service_category"] = df["service_category"].apply(normalize_cell_text)
df["vvip_segment"] = 'Standard'
df["vvip_segment"] = df["vvip_segment"].apply(normalize_cell_text)


df["customer_name"] = df["customer_name"].apply(normalize_trim_text)
df["service_offering_name"] = df["service_offering_name"].apply(normalize_trim_text)

df["channel_name"]= ""
df["deal_id"]= ""
df["customer_alias"]= ""
df["customer_id"]= ""
df["customer_group"]= ""
df["customer_segment_l2"]= ""
df["customer_segment_l3"]= ""
df["deployment_type"]= ""
df["parent_product_category"]= ""
df["partner_name"]= ""
df["product_category"]= ""
df["product_category_code"]= ""
df["product_service_group"]= ""
df["product_version"]= ""
df["revenue_frequency_type"]= ""
df["team_name"]= ""
df["territory_name"]= ""
df["is_hold"]= ""


print(len(df))


16682


In [12]:
set(df["customer_segment_l1"])

{'Bộ Quốc Phòng', 'Ngoài', 'Nội Bộ'}

In [13]:
df.head(5)

,invoice_date,report_month,billing_type,transaction_description,revenue_amount,shared_revenue_amount,actual_revenue_amount,vat_amount,gross_amount,revenue_type,...,parent_product_category,partner_name,product_category,product_category_code,product_service_group,product_version,revenue_frequency_type,team_name,territory_name,is_hold
8832,2023-01-31 00:00:00,01,,,0,0,-498333333.3333333,0,0,,...,,,,,,,,,,
8833,2023-02-28 00:00:00,02,,,0,0,-517291666.6666666,0,0,,...,,,,,,,,,,
8834,2023-03-31 00:00:00,03,,,0,0,-517291666.6666666,0,0,,...,,,,,,,,,,
8835,2023-04-30 00:00:00,04,,,0,0,-555208333.3333333,0,0,,...,,,,,,,,,,
8836,2023-05-31 00:00:00,05,,,0,0,-417083333.33333325,0,0,,...,,,,,,,,,,


In [14]:
df.tail(5)

,invoice_date,report_month,billing_type,transaction_description,revenue_amount,shared_revenue_amount,actual_revenue_amount,vat_amount,gross_amount,revenue_type,...,parent_product_category,partner_name,product_category,product_category_code,product_service_group,product_version,revenue_frequency_type,team_name,territory_name,is_hold
25509,2018-12-31 00:00:00,12,,,0,0,8825481,0,0,,...,,,,,,,,,,
25510,2019-12-31 00:00:00,12,,,0,0,0,0,0,,...,,,,,,,,,,
25511,2019-12-31 00:00:00,12,,,0,0,208800000,0,0,,...,,,,,,,,,,
25512,2019-12-31 00:00:00,12,,,0,0,0,0,0,,...,,,,,,,,,,
25513,2019-12-31 00:00:00,12,,,0,0,8446882.727272727,0,0,,...,,,,,,,,,,


In [15]:
# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open("tmp/" + resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")

fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

✅ Done: revenue_reconciliation.json created
Start crawl :  revenue_reconciliation
revenue_reconciliation
schema_tm_path=./resources/parquet_schema/crm_raw/revenue_reconciliation.json
Replace Upload  s3a://vcs-raw/crm-raw/revenue_reconciliation ./tmp/data/crm_raw/revenue_reconciliation/data_revenue_reconciliation_20260805_140104.parquet
bucket=vcs-raw , key=crm-raw/revenue_reconciliation
objects_to_delete=[{'Key': 'crm-raw/revenue_reconciliation/data_revenue_reconciliation_20260805_104430.parquet'}, {'Key': 'crm-raw/revenue_reconciliation/data_revenue_reconciliation_20260805_105414.parquet'}]
bucket=vcs-raw , key=crm-raw/revenue_reconciliation/data_revenue_reconciliation_20260805_140104.parquet
Uploaded parquet to s3a://vcs-raw/crm-raw/revenue_reconciliation
Uploaded SQL definition


False

In [16]:

# COLUMN_DICT = {
#  "Ngày xuất hóa đơn": "invoice_date",
#     "Tháng": "report_month",
#     "Xuất hóa đơn (HD)/ tạm tính (TT)": "billing_type",
#     "Nội dung đề nghị TT\n(Theo chứng từ gốc)": "transaction_description",
#     "Doanh thu": "revenue_amount",
#     "chia sẻ": "shared_revenue_amount",
#     "doanh thu cuối": "actual_revenue_amount",
#     "VAT": "vat_amount",
#     "Tiền hàng \n(đã bao gồm VAT)": "gross_amount",
#     "DT dòng tiền đều/ DT lên 1 lần": "revenue_type",
#     "Phân loại SP/DV": "service_category",
#     "Nội bộ/ ngoài": "customer_segment_l1",
#     "Phân loại KH": "customer_classification",
#     "Kênh khách hàng": "customer_channel",
#     "Khách hàng": "customer_name",
#     "Segment": "segment",
#     "Phòng": "department_name",
#     "Nhóm khách hàng": "customer_group",
#     "AM": "account_manager_name",
#     "Presale": "presale_name",
#     "SPDV cụ thể": "service_offering_name",
#     "Phân loại DT Cũ/Mới": "revenue_classification",
#     "Phân loại SOC (SOC và non-SOC)": "soc_classification",
#       "DT chia sẻ từ MSS": "shared_revenue_from_mss_amount",
#     "Thị trường": "market_segment",
#     "Mã SPDV": "service_offering_code",
#     "Thị trường/Bắc Nam-SPDV":"market_segment",
#     "Mã SPDV": "service_offering_code",
#     "Phân loại AM (Nội bộ/VVIP/Unname":"am_segment",
#     "TT NAM TT BẮC" : "business_center"
# }

# # filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\KD\Luy_key_3T_2026.xlsx"
# filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\KD\Lũy kế chia sẻ 3T_2026.xlsx"
# resource_name = "revenue_reconciliation"

# df = pd.read_excel(filename, sheet_name="Lũy kế 2026", dtype=str)

# # Remove whitespace ở header nếu có
# df.columns = df.columns.str.strip()
# for col in df.columns:
#     if col not in COLUMN_DICT.keys():
#         print(col)
# # Rename
# df = df[[col for col in df.columns if col in COLUMN_DICT.keys()]]
# df = df.rename(columns=COLUMN_DICT)

# df["source_file"] = "csv\Luy_key_3T_2026.xlsx"
# df["am_segment"] = 'UN'
# df["business_center"] = 'UN'
# df = df.dropna(subset=["invoice_date"])
# df["customer_segment_l1"] = df["customer_segment_l1"].apply(normalize_cell_text)
# df["business_center"] = df["business_center"].apply(normalize_cell_text)
# df["market_segment"] = df["market_segment"].apply(normalize_cell_text)
# df["segment"] = df["segment"].apply(normalize_cell_text)

# df = df.fillna('')
# print(df.columns)
# print(df.dtypes)
# df.head(2)

In [17]:
# # 6. Chuyển sang JSON
# records = df.to_dict(orient="records")
# with open("tmp/" + resource_name +".json", "w", encoding="utf-8") as f:
#     json.dump(records, f, ensure_ascii=False, indent=4)

# print(f"✅ Done: {resource_name}.json created")

# fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

In [18]:

# COLUMN_DICT = {
#  "Ngày xuất hóa đơn": "invoice_date",
#     "Tháng": "report_month",
#     "Xuất hóa đơn (HD)/ tạm tính (TT)": "billing_type",
#     "Nội dung đề nghị TT\n(Theo chứng từ gốc)": "transaction_description",
#     "Nội dung đề nghị TT(Theo chứng từ gốc)":"transaction_description",
#     "Doanh thu": "revenue_amount",
#     "chia sẻ": "shared_revenue_amount",
#     "Doanh thu chia sẻ": "shared_revenue_amount",
#     "doanh thu cuối": "actual_revenue_amount",
#     "Doanh thu cuối": "actual_revenue_amount",
#     "VAT": "vat_amount",
#     "Tiền hàng \n(đã bao gồm VAT)": "gross_amount",
#     "Tiền hàng (đã bao gồm VAT)": "gross_amount",
#     "DT dòng tiền đều/ DT lên 1 lần": "revenue_type",
#     "Phân loại SP/DV": "service_category",
#     "Nội bộ/ ngoài": "customer_segment_l1",
#     "Phân loại KH": "customer_classification",
#     "Kênh khách hàng": "customer_channel",
#     "Khách hàng": "customer_name",
#     "Segment": "segment",
#     "Phòng": "department_name",
#     "Nhóm khách hàng": "customer_group",
#     "AM": "account_manager_name",
#     "AM hiện tại":"account_manager_name",
#     "Presale": "presale_name",
#     "SPDV cụ thể": "service_offering_name",
#     "Phân loại DT Cũ/Mới": "revenue_classification",
#     "Phân loại SOC (SOC và non-SOC)": "soc_classification",
#       "DT chia sẻ từ MSS": "shared_revenue_from_mss_amount",
#     "Thị trường": "market_segment",
#     "Thị trường/Bắc Nam-SPDV":"market_segment",
#     "Mã SPDV": "service_offering_code",
#     "Phân loại AM (Nội bộ/VVIP/Unname":"am_segment",
#     "TT NAM TT BẮC" : "business_center"
# }

# # filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\KD\TĐ_T4T5 CHOT_step_2_MSS_share_TI.xlsx"
# filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\KD\TĐ_T4T5 CHOT_step_2_MSS_share_TI.xlsx"
# resource_name = "revenue_reconciliation"

# df = pd.read_excel(filename, sheet_name="Sheet1", dtype=str)

# # Remove whitespace ở header nếu có
# df.columns = df.columns.str.strip()
# for col in df.columns:
#     if col not in COLUMN_DICT.keys():
#         print(col)
# # Rename
# df = df[[col for col in df.columns if col in COLUMN_DICT.keys()]]
# df = df.rename(columns=COLUMN_DICT)

# df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")
# df = df.dropna(subset=["invoice_date"])
# df["customer_segment_l1"] = df["customer_segment_l1"].apply(normalize_cell_text)
# df["business_center"] = df["business_center"].apply(normalize_cell_text)
# df["market_segment"] = df["market_segment"].apply(normalize_cell_text)
# df["segment"] = df["segment"].apply(normalize_cell_text)

# df["source_file"] = "KD\Luy_Ke_Chia_Se_5T_2026.xlsx"
# df = df.fillna('')
# print(df.columns)
# print(df.dtypes)
# df.head(2)

In [19]:
# # 6. Chuyển sang JSON
# records = df.to_dict(orient="records")
# with open("tmp/" + resource_name +".json", "w", encoding="utf-8") as f:
#     json.dump(records, f, ensure_ascii=False, indent=4)

# print(f"✅ Done: {resource_name}.json created")

# fetch_resource_name_freshwork(resource_name, records, crawl_mode="modified_and_new")

In [20]:

# COLUMN_DICT = {
#  "Ngày xuất hóa đơn": "invoice_date",
#     "Tháng": "report_month",
#     "Xuất hóa đơn (HD)/ tạm tính (TT)": "billing_type",
#     "Nội dung đề nghị TT\n(Theo chứng từ gốc)": "transaction_description",
#     "Nội dung đề nghị TT(Theo chứng từ gốc)":"transaction_description",
#     "Doanh thu": "revenue_amount",
#     "chia sẻ": "shared_revenue_amount",
#     "Doanh thu chia sẻ": "shared_revenue_amount",
#     "doanh thu cuối": "actual_revenue_amount",
#     "Doanh thu cuối": "actual_revenue_amount",
#     "VAT": "vat_amount",
#     "Tiền hàng \n(đã bao gồm VAT)": "gross_amount",
#     "Tiền hàng (đã bao gồm VAT)": "gross_amount",
#     "DT dòng tiền đều/ DT lên 1 lần": "revenue_type",
#     "Phân loại SP/DV": "service_category",
#     "Nội bộ/ ngoài": "customer_segment_l1",
#     "Phân loại KH": "customer_classification",
#     "Kênh khách hàng": "customer_channel",
#     "Khách hàng": "customer_name",
#     "Segment": "segment",
#     "Phòng": "department_name",
#     "Nhóm khách hàng": "customer_group",
#     "AM": "account_manager_name",
#     "AM hiện tại":"account_manager_name",
#     "Presale": "presale_name",
#     "SPDV cụ thể": "service_offering_name",
#     "Phân loại DT Cũ/Mới": "revenue_classification",
#     "Phân loại SOC (SOC và non-SOC)": "soc_classification",
#       "DT chia sẻ từ MSS": "shared_revenue_from_mss_amount",
#     "Thị trường": "market_segment",
#     "Thị trường/Bắc Nam-SPDV":"market_segment",
#     "Mã SPDV": "service_offering_code",
#     "Phân loại AM (Nội bộ/VVIP/Unname":"am_segment",
#     "TT NAM TT BẮC" : "business_center"
# }

# filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\KD\TĐ_THANG 6_step_2_MSS_share_TI.xlsx"
# resource_name = "revenue_reconciliation"

# df = pd.read_excel(filename, sheet_name="Sheet1", dtype=str)

# # Remove whitespace ở header nếu có
# df.columns = df.columns.str.strip()
# for col in df.columns:
#     if col not in COLUMN_DICT.keys():
#         print(col)
# # Rename
# df = df[[col for col in df.columns if col in COLUMN_DICT.keys()]]
# df = df.rename(columns=COLUMN_DICT)

# df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")
# df = df.dropna(subset=["invoice_date"])

# df["customer_segment_l1"] = df["customer_segment_l1"].apply(normalize_cell_text)
# df["business_center"] = df["business_center"].apply(normalize_cell_text)
# df["market_segment"] = df["market_segment"].apply(normalize_cell_text)
# df["segment"] = df["segment"].apply(normalize_cell_text)

# df["source_file"] = "KD\TĐ_THANG 6_step_2_MSS_share_TI.xlsx"
# df = df.fillna('')
# print(df.columns)
# print(df.dtypes)
# print(len(df))

In [21]:
# df.head(2)

In [22]:
# # 6. Chuyển sang JSON
# records = df.to_dict(orient="records")
# with open("tmp/" + resource_name +".json", "w", encoding="utf-8") as f:
#     json.dump(records, f, ensure_ascii=False, indent=4)

# print(f"✅ Done: {resource_name}.json created")

# fetch_resource_name_freshwork(resource_name, records, crawl_mode="modified_and_new")

In [23]:

# COLUMN_DICT = {
#  "Ngày xuất hóa đơn": "invoice_date",
#     "Tháng": "report_month",
#     "Xuất hóa đơn (HD)/ tạm tính (TT)": "billing_type",
#     "Nội dung đề nghị TT\n(Theo chứng từ gốc)": "transaction_description",
#     "Nội dung đề nghị TT(Theo chứng từ gốc)":"transaction_description",
#     # "Doanh thu": "revenue_amount",
#     # "chia sẻ": "shared_revenue_amount",
#     # "Doanh thu chia sẻ": "shared_revenue_amount",
#     # "doanh thu cuối": "actual_revenue_amount",
#     "Doanh thu": "actual_revenue_amount",
#     "VAT": "vat_amount",
#     "Tiền hàng \n(đã bao gồm VAT)": "gross_amount",
#     "Tiền hàng (đã bao gồm VAT)": "gross_amount",
#     "DT dòng tiền đều/ DT lên 1 lần": "revenue_type",
#     "Phân loại SP/DV": "service_category",
#     "Nội bộ/ ngoài": "customer_segment_l1",
#     "Phân loại KH": "customer_classification",
#     "Kênh khách hàng": "customer_channel",
#     "Khách hàng": "customer_name",
#     "Segment": "segment",
#     "Phòng": "department_name",
#     "Nhóm khách hàng": "customer_group",
#     "AM": "account_manager_name",
#     # "AM hiện tại":"account_manager_name",
#     "AM hiện tại .1":"account_manager_name",
#     "Presale": "presale_name",
#     "SPDV cụ thể": "service_offering_name",
#     "Phân loại DT Cũ/Mới": "revenue_classification",
#     "Phân loại SOC (SOC và non-SOC)": "soc_classification",
#       "DT chia sẻ từ MSS": "shared_revenue_from_mss_amount",
#     "Thị trường": "market_segment",
#     "Thị trường/Bắc Nam-SPDV":"market_segment",
#     "Mã SPDV": "service_offering_code",
#     "Phân loại AM (Nội bộ/VVIP/Unname":"am_segment",
#     "TT NAM TT BẮC" :  "business_center",
#     "TT NAM \nTT BẮC": "business_center",
#     "VIP/ VVIP theo TTr":"vvip_segment",
#     "Ghi chú": "note",
# }

# # filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\KD\LK 6 tháng TĐ&VCS (1).xlsx"
# filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\KD\DOANH THU 6 T VCS_TĐ_CHỐT_CHUẨN HÓA.xlsx"

# resource_name = "revenue_reconciliation"

# df = pd.read_excel(filename, sheet_name="TĐ_ LK 6 tháng", dtype=str, header=1)

# # Remove whitespace ở header nếu có
# df.columns = df.columns.str.strip()
# for col in df.columns:
#     if col not in COLUMN_DICT.keys():
#         print(col)
# # Rename
# df = df[[col for col in df.columns if col in COLUMN_DICT.keys()]]
# df = df.rename(columns=COLUMN_DICT)

# df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")
# df = df.dropna(subset=["invoice_date"])

# df['shared_revenue_amount'] = 0
# df['revenue_amount'] = df['actual_revenue_amount'] 
# df["customer_segment_l1"] = df["customer_segment_l1"].apply(normalize_cell_text)
# df["business_center"] = df["business_center"].apply(normalize_cell_text)
# df["market_segment"] = df["market_segment"].apply(normalize_cell_text)
# df["segment"] = df["segment"].apply(normalize_cell_text)
# df["service_category"] = df["service_category"].apply(normalize_cell_text)
# df["vvip_segment"] = df["vvip_segment"].apply(normalize_cell_text)

# df["customer_name"] = df["customer_name"].apply(normalize_trim_text)
# df["service_offering_name"] = df["service_offering_name"].apply(normalize_trim_text)


# df["source_file"] = r"finance-data\KD\DOANH THU 6 T VCS_TĐ_CHỐT_CHUẨN HÓA.xlsx"
# df = df.fillna('')
# print(df.columns)
# print(df.dtypes)
# print(len(df))

In [24]:
# set(df["customer_segment_l1"])

In [25]:
# df.head(2)

In [26]:
# df.tail(2)

In [27]:
# # 6. Chuyển sang JSON
# records = df.to_dict(orient="records")
# with open("tmp/" + resource_name +".json", "w", encoding="utf-8") as f:
#     json.dump(records, f, ensure_ascii=False, indent=4)

# print(f"✅ Done: {resource_name}.json created")

# fetch_resource_name_freshwork(resource_name, records, crawl_mode="modified_and_new")

In [28]:

COLUMN_DICT = {
 "Ngày xuất hóa đơn": "invoice_date",
    "Tháng": "report_month",
    # "Xuất hóa đơn (HD)/ tạm tính (TT)": "billing_type",
    "Nội dung đề nghị TT\n(Theo chứng từ gốc)": "transaction_description",
    "Nội dung đề nghị TT(Theo chứng từ gốc)":"transaction_description",
    # "Doanh thu": "revenue_amount",
    # "chia sẻ": "shared_revenue_amount",
    # "Doanh thu chia sẻ": "shared_revenue_amount",
    # "doanh thu cuối": "actual_revenue_amount",
    "Doanh thu": "actual_revenue_amount",
    "VAT": "vat_amount",
    "Tiền hàng \n(đã bao gồm VAT)": "gross_amount",
    "Tiền hàng (đã bao gồm VAT)": "gross_amount",
    # "DT dòng tiền đều/ DT lên 1 lần": "revenue_type",
    "Phân loại SP/DV": "service_category",
    # "Nội bộ/ ngoài": "customer_segment_l1",
    "Phân loại KH": "customer_classification",
    "Kênh khách hàng": "customer_channel",
    "Khách hàng": "customer_name",
    "Segment": "segment",
    "Phòng": "department_name",
    "Nhóm khách hàng": "customer_group",
    "AM": "account_manager_name",
    # "AM hiện tại":"account_manager_name",
    "AM hiện tại .1":"account_manager_name",
    "Presale": "presale_name",
    "SPDV cụ thể": "service_offering_name",
    "Phân loại DT Cũ/Mới": "revenue_classification",
    # "Phân loại SOC (SOC và non-SOC)": "soc_classification",
      "DT chia sẻ từ MSS": "shared_revenue_from_mss_amount",
    "Thị trường": "market_segment",
    "Thị trường/Bắc Nam-SPDV":"market_segment",
    "Mã SPDV": "service_offering_code",
    "Phân loại AM (Nội bộ/VVIP/Unname":"am_segment",
    "TT NAM TT BẮC" :  "business_center",
    "TT NAM \nTT BẮC": "business_center",
    "VIP/ VVIP theo TTr":"vvip_segment",
    "Ghi chú": "note",
    "billing_type":"billing_type",
    "channel_name":"channel_name",
    "deal_id":"deal_id",
    "customer_alias":"customer_alias",
    "customer_id":"customer_id",
    "customer_group":"customer_group",
    "customer_segment_l1":"customer_segment_l1",
    "customer_segment_l2":"customer_segment_l2",
    "customer_segment_l3":"customer_segment_l3",
    "deployment_type":"deployment_type",
    "parent_product_category":"parent_product_category",
    "partner_name":"partner_name",
    "product_category":"product_category",
    "product_category_code":"product_category_code",
    "product_service_group":"product_service_group",
    "product_version":"product_version",
    "revenue_frequency_type":"revenue_frequency_type",
    "revenue_type":"revenue_type",
    "soc_classification":"soc_classification",
    "team_name":"team_name",
    "territory_name":"territory_name",
    "is_hold":"is_hold",
}

# filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\KD\LK 6 tháng TĐ&VCS (1).xlsx"
filename = r"C:\Users\namtv40\Documents\AI Chatbot\KD-Data\ĐHKD_DataRequest_28072026_mapped_dimensions.xlsx"

resource_name = "revenue_reconciliation"

df = pd.read_excel(filename, sheet_name="Doanh thu thực hiện_2024_Mapped", dtype=str, header=0)

# Remove whitespace ở header nếu có
df.columns = df.columns.str.strip()
for col in df.columns:
    if col not in COLUMN_DICT.keys():
        print(col)
# Rename
df = df[[col for col in df.columns if col in COLUMN_DICT.keys()]]
df = df.rename(columns=COLUMN_DICT)

df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")
df = df.dropna(subset=["invoice_date"])

df['shared_revenue_amount'] = 0
df['revenue_amount'] = df['actual_revenue_amount'] 
df["customer_segment_l1"] = df["customer_segment_l1"].apply(normalize_cell_text)
df["business_center"] = ""
df["market_segment"] = df["market_segment"].apply(normalize_cell_text)
df["segment"] = df["segment"].apply(normalize_cell_text)
df["service_category"] = df["service_category"].apply(normalize_cell_text)
df["vvip_segment"] = ""
df['shared_revenue_amount'] = 0

df["customer_name"] = df["customer_name"].apply(normalize_trim_text)
df["service_offering_name"] = df["service_offering_name"].apply(normalize_trim_text)


df["source_file"] = r"KD-Data\ĐHKD_DataRequest_28072026_mapped_dimensions.xlsx"
df = df.fillna('')
print(df.columns)
print(df.dtypes)
print(len(df))

Năm
Xuất hóa đơn (HD)/ tạm tính (TT)
DT dòng tiền đều/ DT lên 1 lần
Nội bộ/ ngoài
Phân loại SOC (SOC và non-SOC)
TT Nam/Bắc
Index(['invoice_date', 'report_month', 'transaction_description',
       'actual_revenue_amount', 'vat_amount', 'gross_amount',
       'service_category', 'customer_classification', 'customer_channel',
       'customer_name', 'segment', 'department_name', 'customer_group',
       'account_manager_name', 'presale_name', 'service_offering_name',
       'revenue_classification', 'shared_revenue_from_mss_amount',
       'market_segment', 'service_offering_code', 'note', 'billing_type',
       'channel_name', 'deal_id', 'customer_alias', 'customer_id',
       'customer_group', 'customer_segment_l1', 'customer_segment_l2',
       'customer_segment_l3', 'deployment_type', 'parent_product_category',
       'partner_name', 'product_category', 'product_category_code',
       'product_service_group', 'product_version', 'revenue_frequency_type',
       'revenue_type', 'soc_cl

In [29]:
df.head(2)

,invoice_date,report_month,transaction_description,actual_revenue_amount,vat_amount,gross_amount,service_category,customer_classification,customer_channel,customer_name,...,revenue_type,soc_classification,team_name,territory_name,is_hold,shared_revenue_amount,revenue_amount,business_center,vvip_segment,source_file
0,2026-03-31 00:00:00,T3,Dịch vụ giám sát ATTT cho VPBank giai đoạn từ ...,-818775246.342466,0,-818775246.342466,Mss,Khách hàng - Trực tiếp,Khách hàng - trực tiếp,VPBank,...,Doanh thu gia hạn,SOC,,Miền Bắc,False,0,-818775246.342466,,,KD-Data\ĐHKD_DataRequest_28072026_mapped_dimen...
1,2026-03-31 00:00:00,T3,Dịch vụ Giám sát an toàn thông tin 24/7 theo h...,360937500,0,360937500,Mss,Khách hàng - Trực tiếp,Khách hàng - trực tiếp,VPBank,...,Doanh thu gia hạn,SOC,,Miền Bắc,False,0,360937500,,,KD-Data\ĐHKD_DataRequest_28072026_mapped_dimen...


In [30]:
df.tail(2)

,invoice_date,report_month,transaction_description,actual_revenue_amount,vat_amount,gross_amount,service_category,customer_classification,customer_channel,customer_name,...,revenue_type,soc_classification,team_name,territory_name,is_hold,shared_revenue_amount,revenue_amount,business_center,vvip_segment,source_file
15216,2026-06-30 00:00:00,T6,Dịch vụ giám sát an toàn thông tin mạng 24/7 t...,49333333.33333333,0,49333333.33333333,Mss,Khách hàng - trực tiếp,Khách hàng - trực tiếp,FECREDIT,...,Doanh thu gia hạn,SOC,,Miền Bắc,True,0,49333333.33333333,,,KD-Data\ĐHKD_DataRequest_28072026_mapped_dimen...
15217,2026-06-30 00:00:00,T6,Dịch vụ giám sát an toàn thông tin mạng 24/7 t...,259000000,0,259000000,Mss,Khách hàng - trực tiếp,Khách hàng - trực tiếp,FECREDIT,...,Doanh thu gia hạn,SOC,,Miền Bắc,True,0,259000000,,,KD-Data\ĐHKD_DataRequest_28072026_mapped_dimen...


In [31]:
# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open("tmp/" + resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")

fetch_resource_name_freshwork(resource_name, records, crawl_mode="modified_and_new")

C:\Users\namtv40\AppData\Local\Temp\ipykernel_48976\3701229883.py:2: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  records = df.to_dict(orient="records")


✅ Done: revenue_reconciliation.json created
Start crawl :  revenue_reconciliation
revenue_reconciliation
schema_tm_path=./resources/parquet_schema/crm_raw/revenue_reconciliation.json
Add Upload  s3a://vcs-raw/crm-raw/revenue_reconciliation ./tmp/data/crm_raw/revenue_reconciliation/data_revenue_reconciliation_20260805_140207.parquet
bucket=vcs-raw , key=crm-raw/revenue_reconciliation/data_revenue_reconciliation_20260805_140207.parquet
Uploaded parquet to s3a://vcs-raw/crm-raw/revenue_reconciliation
Uploaded SQL definition


False